This is the solution of F21BC Week 4 Lab with keras version code - Part 2 

a) Check the code from mnist_cnn.py and run it.

b) Add a MaxPool1D layer after the first convolutional layer, and a dropout of 25% after it. Also, change the optimization algorithm to “adam”.

In [ ]:
from __future__ import print_function
import keras
from keras.datasets import mnist
from keras.models import Sequential

# Task b: import Reshape to transfer 2D data into 1D, then we can use MaxPooling1D
from keras.layers import Dense, Dropout, Flatten, Reshape

# Task b: import MaxPooling1D package
from keras.layers import Conv2D, MaxPooling2D, MaxPooling1D
from keras import backend as K

batch_size = 128
num_classes = 10
epochs = 12

# input image dimensions
img_rows, img_cols = 28, 28

# the data, split between train and test sets
(x_train, y_train), (x_test, y_test) = mnist.load_data()

if K.image_data_format() == 'channels_first':
    x_train = x_train.reshape(x_train.shape[0], 1, img_rows, img_cols)
    x_test = x_test.reshape(x_test.shape[0], 1, img_rows, img_cols)
    input_shape = (1, img_rows, img_cols)
else:
    x_train = x_train.reshape(x_train.shape[0], img_rows, img_cols, 1)
    x_test = x_test.reshape(x_test.shape[0], img_rows, img_cols, 1)
    input_shape = (img_rows, img_cols, 1)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
print('x_train shape:', x_train.shape)
print(x_train.shape[0], 'train samples')
print(x_test.shape[0], 'test samples')

# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3),
                 activation='relu',
                 input_shape=input_shape))

# Task b: Add a MaxPoo1D layer after the first convolutional layer, and a dropout of 25% after it.
# Output size = (input size - kernel size) + 2 * padding + 1
#        = (28 - 3) + 2 * 0 + 1 = 26
# Reshape from (batch, 26, 26, 32) to (batch, 676, 32) for 1D pooling
# (note: this is a 2D CNN working with MNIST images, so MaxPool2D is appropriate, not MaxPool1D)
# model.add(Reshape((26*26, 32)))
# model.add(MaxPooling1D(pool_size=2))
# model.add(Dropout(0.25))
# # Reshape back to 2D format for next Conv2D layer
# model.add(Reshape((13, 26, 32)))

# Task b(new): Add a MaxPoo2D layer after the first convolutional layer, and a dropout of 25% after it.
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

# Task b: change the optimization algorithm to “adam”.
model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer=keras.optimizers.Adam(),
              metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(x_test, y_test))
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])
